# Reconstructing pickleball flights with bounces and physics

Companion notebook for the condados.ai post. It refits one flight (52) from its pixels, checks it against the net, and loads the saved results for all 80 flights. CPU only.

Run from the repository root after `uv sync`, or in Colab after the first cell.

In [ ]:
# Colab: uncomment to get the code and its dependencies
# !git clone --depth 1 https://github.com/CondadosAI/condados-lab && cd condados-lab/pickleball-3d-ball
# !pip install -q 'numpy>=2' 'scipy>=1.14'
import os, sys, json
if os.path.basename(os.getcwd()) != 'pickleball-3d-ball':
    os.chdir('pickleball-3d-ball')
sys.path.insert(0, '.')
import numpy as np
from fit import load, fit_flight, trajectory

## One flight's pixels

Flight 52 is one groundstroke: a bounce, a paddle hit, a bounce on the other side of the net. Each detection is a pixel and a time.

In [ ]:
camera, flights, data = load()          # the radial camera, 80 flights
f = flights[52]
print(len(f.times), 'detections from', f.times[0], 'to', f.times[-1], 's')
print('first bounce', f.bounce1_t, 's, detected hit', f.hit_t, 's, second bounce', f.bounce2_t, 's')
print('camera centre in court metres:', camera.position.round(2))

## A pixel is a ray

`camera.ray` turns a pixel into a line from the camera centre. The ball is somewhere on it; one camera cannot say where.

In [ ]:
origin, direction = camera.ray(np.asarray(f.pixels[:1]))
for depth in (5, 10, 15):
    p = (origin + depth * direction).reshape(-1)
    print(f'{depth:>2} m along the ray: x={p[0]:.2f}  y={p[1]:.2f}  height={p[2]:.2f} m')

## The fit

Gravity 9.81, drag 0.05 per metre, both bounces on the floor, the paddle time free within 0.15 s of the detected hit.

In [ ]:
fit = fit_flight(camera, f, gravity=9.81, drag=0.05, fit_drag=False, hit_window_s=0.15)
print('median pixel residual', round(fit['median_px'], 2), 'px over', fit['observations'], 'detections')
print('apex', round(fit['apex_m'], 3), 'm; paddle contact at', round(fit['paddle_height_m'], 2), 'm')
print('speed off the paddle', round(fit['paddle_speed_m_s'], 2), 'm/s =', round(fit['paddle_speed_m_s'] * 2.23694, 1), 'mph')
print('paddle time moved', round((fit['hit_t'] - f.hit_t) * data['fps'], 1), 'frames after the detected hit')

Pinned to the detected hit instead, the same flight explains its pixels worse:

In [ ]:
pinned = fit_flight(camera, f, gravity=9.81, drag=0.05, fit_drag=False)
print('median pixel residual with the detected hit time:', round(pinned['median_px'], 2), 'px')

## The net

The second bounce is on the other side of the net, so the arc crossed it. Its centre must be at least one ball radius (3.6 cm) above the tape.

In [ ]:
net = json.load(open('output/net.json'))
print(net['rows'][52])
print({k: net[k] for k in ('flights', 'cross_the_net', 'clear_it', 'miss_within_noise', 'miss_beyond_noise')})
bad = [r for r in net['rows'] if r['clearance_m'] < -0.05 and r['median_px'] <= 3]
print(len(bad), 'flights with a good pixel fit pass under the net:', [(r['flight'], r['clearance_m']) for r in bad])

## Gravity left free, all 80 flights

`check_gravity.py` (about 20 minutes on 20 cores) fits every flight with g free. The summary:

In [ ]:
g = json.load(open('output/gravity.json'))
for cam, r in g['real'].items():
    print(f"real, {cam:7s}: median {r['median']} ± {r['se']} (bootstrap SE), IQR {r['iqr']}, n={r['n']}, residual {r['median_residual_px']} px")
for name, r in g['synthetic'].items():
    print(f"synthetic, {name}: {r['median']} ± {r['se']}")
free = fit_flight(camera, f, fit_gravity=True, drag=0.05, fit_drag=False, hit_window_s=0.15)
print('flight 52 alone, g free:', round(free['g_m_s2'], 2), 'm/s^2  (one flight is not a measurement)')

## The scale one camera cannot see

Scale the fitted path about the camera centre: every point stays on its ray, so every pixel stays put, and the bounce leaves the floor.

In [ ]:
t = np.linspace(fit['bounce1_t'], fit['bounce2_t'], 5)
path = trajectory(fit, t)
c = camera.position
for s in (1.0, 0.8, 1.2):
    scaled = c + s * (path - c)
    moved = np.abs(camera.to_image(scaled) - camera.to_image(path)).max()
    print(f'scale {s}: first bounce height {scaled[0, 2]:+.2f} m, pixels moved by at most {moved:.1e} px')